In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from datetime import datetime
import altair as alt

In [2]:
df = pd.read_csv('Crash_Reporting_-_Drivers_Data.csv')

C:\Users\dku19\AppData\Local\Temp\ipykernel_6480\1513844872.py:1: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('Crash_Reporting_-_Drivers_Data.csv')


In [3]:
df['Crash Date/Time'] = pd.to_datetime(df['Crash Date/Time'])

# Extract the date part
df['Date'] = df['Crash Date/Time'].dt.date

# Extract the time part
df['Time'] = df['Crash Date/Time'].dt.time


C:\Users\dku19\AppData\Local\Temp\ipykernel_6480\4020282977.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['Crash Date/Time'] = pd.to_datetime(df['Crash Date/Time'])


In [4]:
df['Injury Severity'] = df['Injury Severity'].str.lower()
df_new = df.copy()
list(set(df['Injury Severity']))

['no apparent injury',
 'fatal injury',
 'suspected minor injury',
 'possible injury',
 nan,
 'suspected serious injury']

In [5]:
df['Weather'] = df['Weather'].str.lower()
df['Weather']

0                                     clear
1                                     clear
2                                     clear
3                                     clear
4                                     clear
                        ...                
192492                               cloudy
192493                                 snow
192494                               cloudy
192495    freezing rain or freezing drizzle
192496                                clear
Name: Weather, Length: 192497, dtype: object

In [6]:
df = df.dropna(subset=['Injury Severity'])
list(set(df['Weather']))

['foggy',
 'snow',
 'rain',
 'raining',
 'unknown',
 'fog, smog, smoke',
 'other',
 'blowing snow',
 'severe crosswinds',
 'severe winds',
 'sleet',
 'blowing sand, soil, dirt',
 'cloudy',
 'sleet or hail',
 'freezing rain or freezing drizzle',
 'clear',
 nan,
 'wintry mix']

In [7]:
df['Driver At Fault']

0              No
1             Yes
2             Yes
3         Unknown
4              No
           ...   
192492        Yes
192493         No
192494        Yes
192495         No
192496        Yes
Name: Driver At Fault, Length: 191242, dtype: object

In [8]:
hazard_mapping = {
    'clear': 'Not Hazardous',
    'cloudy': 'Not Hazardous',
    'rain': 'Not Hazardous',
    'raining': 'Not Hazardous',
    'other': 'Not Hazardous',
    'unknown': 'Not Hazardous',
    'foggy': 'Moderate Hazardous',
    'snow': 'Moderate Hazardous',
    'sleet': 'Moderate Hazardous',
    'wintry mix': 'Moderate Hazardous',
    'blowing sand, soil, dirt': 'Moderate Hazardous',
    'fog, smog, smoke': 'Moderate Hazardous',
    'sleet or hail': 'Moderate Hazardous',
    'blowing snow': 'Very Hazardous',
    'severe winds': 'Very Hazardous',
    'freezing rain or freezing drizzle': 'Very Hazardous',
    'severe crosswinds': 'Very Hazardous'
}

In [9]:
df['hazard_level'] = df['Weather'].map(hazard_mapping)


# # Fill NaN with 'Unknown' or 'Not Hazardous' based on your preference
df['hazard_level'].fillna('Unknown', inplace=True)

C:\Users\dku19\AppData\Local\Temp\ipykernel_6480\3819056346.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['hazard_level'].fillna('Unknown', inplace=True)


In [10]:
hazard_injury_df = df.groupby(['hazard_level', 'Injury Severity', 'Driver At Fault']).size().reset_index(name='count')
hazard_injury_df

,hazard_level,Injury Severity,Driver At Fault,count
0,Moderate Hazardous,fatal injury,Unknown,1
1,Moderate Hazardous,fatal injury,Yes,1
2,Moderate Hazardous,no apparent injury,No,1018
3,Moderate Hazardous,no apparent injury,Unknown,64
4,Moderate Hazardous,no apparent injury,Yes,1439
5,Moderate Hazardous,possible injury,No,149
6,Moderate Hazardous,possible injury,Unknown,3
7,Moderate Hazardous,possible injury,Yes,120
8,Moderate Hazardous,suspected minor injury,No,113
9,Moderate Hazardous,suspected minor injury,Unknown,2


In [11]:
total_by_hazard = hazard_injury_df.groupby('hazard_level')['count'].transform('sum')

# Calculate the proportion by dividing count by the total for that hazard level
hazard_injury_df['proportion'] = round(hazard_injury_df['count'] / total_by_hazard, 3)

hazard_injury_df['total_in_hazard_level'] = total_by_hazard

hazard_injury_df = hazard_injury_df[hazard_injury_df['hazard_level'] != 'Unknown']
hazard_injury_df = hazard_injury_df[hazard_injury_df['Driver At Fault'] != 'Unknown']


hazard_injury_df

,hazard_level,Injury Severity,Driver At Fault,count,proportion,total_in_hazard_level
1,Moderate Hazardous,fatal injury,Yes,1,0.000,3040
2,Moderate Hazardous,no apparent injury,No,1018,0.335,3040
4,Moderate Hazardous,no apparent injury,Yes,1439,0.473,3040
5,Moderate Hazardous,possible injury,No,149,0.049,3040
7,Moderate Hazardous,possible injury,Yes,120,0.039,3040
8,Moderate Hazardous,suspected minor injury,No,113,0.037,3040
10,Moderate Hazardous,suspected minor injury,Yes,108,0.036,3040
11,Moderate Hazardous,suspected serious injury,No,9,0.003,3040
12,Moderate Hazardous,suspected serious injury,Yes,13,0.004,3040
13,Not Hazardous,fatal injury,No,30,0.000,174474


In [12]:
hazard_order = ['Not Hazardous', 'Moderate Hazardous', 'Very Hazardous']

bar_plot = alt.Chart(hazard_injury_df).mark_bar().encode(
    x = alt.X('hazard_level:N', 
              axis=alt.Axis(labelAngle=0, title = "Hazard Level"),
            scale=alt.Scale(domain=hazard_order)),        
    y = alt.Y('proportion:Q',   axis=alt.Axis(title='Proportion')),         
    color='Injury Severity:N',    
    xOffset='Injury Severity:N',
    tooltip = ['Injury Severity', 'count', 'total_in_hazard_level', 'proportion'],
    column='Driver At Fault:N'
).properties(
    title='Proportion of Injury Severity by Weather Hazard Level and Driver At Fault'
)

bar_plot

C:\Users\dku19\anaconda3\Lib\site-packages\altair\utils\core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
C:\Users\dku19\anaconda3\Lib\site-packages\altair\utils\core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
C:\Users\dku19\anaconda3\Lib\site-packages\altair\utils\core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)


alt.Chart(...)

In [13]:
bar_plot.save('bar_plot.html')

C:\Users\dku19\anaconda3\Lib\site-packages\altair\utils\core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
C:\Users\dku19\anaconda3\Lib\site-packages\altair\utils\core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
C:\Users\dku19\anaconda3\Lib\site-packages\altair\utils\core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)


In [14]:
from shapely.geometry import Point

In [15]:
df['geometry'] = df.apply(lambda row: Point(row['Longitude'], row['Latitude']), axis=1)


In [16]:
df

,Report Number,Local Case Number,Agency Name,ACRS Report Type,Crash Date/Time,Route Type,Road Name,Cross-Street Name,Off-Road Description,Municipality,...,Vehicle Year,Vehicle Make,Vehicle Model,Latitude,Longitude,Location,Date,Time,hazard_level,geometry
0,DM8479000T,210020119,Takoma Park Police Depart,Property Damage Crash,2021-05-27 19:40:00,NaN,NaN,NaN,IN PARKING LOT,NaN,...,2017,HINO,TWK,38.987657,-76.987545,"(38.98765667, -76.987545)",2021-05-27,19:40:00,Not Hazardous,POINT (-76.987545 38.98765667)
1,MCP2970000R,15045937,MONTGOMERY,Property Damage Crash,2015-09-11 13:29:00,NaN,NaN,NaN,Parking Lot: \n2525 Ennalls Ave,NaN,...,2012,TOYOTA,SU,39.039917,-77.053649,"(39.03991652, -77.05364898)",2015-09-11,13:29:00,Not Hazardous,POINT (-77.05364898 39.03991652)
2,MCP20160036,180040948,Montgomery County Police,Property Damage Crash,2018-08-17 14:25:00,NaN,NaN,NaN,PARKING LOT OF 16246 FREDERICK RD,NaN,...,2015,MAZD,TK,38.743373,-77.546997,"(38.743373, -77.54699707)",2018-08-17,14:25:00,Not Hazardous,POINT (-77.54699707 38.743373)
3,EJ7879003C,230048975,Gaithersburg Police Depar,Injury Crash,2023-08-11 18:00:00,NaN,NaN,NaN,1 N SUMMIT DRIVE,NaN,...,2018,RAM,TK,39.145873,-77.191940,"(39.14587303, -77.19194047)",2023-08-11,18:00:00,Not Hazardous,POINT (-77.19194047 39.14587303)
4,MCP2967004Y,230070277,Montgomery County Police,Property Damage Crash,2023-12-06 18:42:00,Maryland (State),CONNECTICUT AVE,BALTIMORE ST,NaN,KENSINGTON,...,2017,AUDI,A3,39.025170,-77.076333,"(39.02517017, -77.07633333)",2023-12-06,18:42:00,Not Hazardous,POINT (-77.07633333 39.02517017)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
192492,MCP137801BV,250004908,MONTGOMERY,Property Damage Crash,2025-02-03 12:38:00,Maryland (State) Route,NaN,CAMP ST,NaN,NaN,...,2015,TOYOTA,CAMRY,39.156678,-77.157089,"(39.15667823, -77.15708915)",2025-02-03,12:38:00,Not Hazardous,POINT (-77.15708915 39.15667823)
192493,DM84390042,250006269,TAKOMA,Property Damage Crash,2025-02-11 18:15:00,Municipality Route,BOYD AVE,JACKSON AVE,NaN,NaN,...,2021,FORD,EXPLORER,38.978800,-77.001527,"(38.97879989, -77.00152685)",2025-02-11,18:15:00,Moderate Hazardous,POINT (-77.00152685 38.97879989)
192494,MCP137801BZ,250005469,MONTGOMERY,Property Damage Crash,2025-02-06 15:34:00,County Route,BRENISH DR,KILDONAN DR,NaN,NaN,...,2017,THOMAS BUILT,HDX,39.178215,-77.152550,"(39.17821481, -77.15255031)",2025-02-06,15:34:00,Not Hazardous,POINT (-77.15255031 39.17821481)
192495,MCP31090095,250005397,MONTGOMERY,Injury Crash,2025-02-05 22:27:00,County Route,EAST VILLAGE AVE (WB/L),NaN,NaN,NaN,...,2006,DODGE,CHARGER,39.185790,-77.170561,"(39.18579008, -77.17056071)",2025-02-05,22:27:00,Very Hazardous,POINT (-77.17056071 39.18579008)


In [17]:
import geopandas as gpd

gdf = gpd.GeoDataFrame(df, crs="EPSG:4326", geometry='geometry')

gdf['Crash Year'] = gdf['Crash Date/Time'].dt.year

gdf['Crash Year']

0         2021
1         2015
2         2018
3         2023
4         2023
          ... 
192492    2025
192493    2025
192494    2025
192495    2025
192496    2025
Name: Crash Year, Length: 191242, dtype: int32

In [18]:
gdf['Latitude'] = pd.to_numeric(gdf['Latitude'], errors='coerce')
gdf['Longitude'] = pd.to_numeric(gdf['Longitude'], errors='coerce')
print(gdf[['Latitude', 'Longitude']].dtypes)

Latitude     float64
Longitude    float64
dtype: object


In [19]:
import folium


import plotly.express as px

gdf_2024_serious = gdf[(gdf['Crash Year'] == 2024) & (gdf['Injury Severity'] == 'suspected serious injury')]
gdf_2024_minor = gdf[(gdf['Crash Year'] == 2024) & (gdf['Injury Severity'] == 'suspected minor injury')]
gdf_2024_possible = gdf[(gdf['Crash Year'] == 2024) & (gdf['Injury Severity'] == 'possible injury')]


# m = maryland.explore(
#     tooltip = ['COUNTY', 'DISTRICT'],
#     name = 'Washington Area'
# )


gdf_2024_serious = gdf_2024_serious.explore(
    color = 'red',
    marker_kwds = dict(radius = 3, fill = True),
    tooltip = ['Road Name', 'Weather', 'Injury Severity'],
    popup = ['Crash Date/Time', 'Cross-Street Name', 'Off-Road Description', 'Vehicle Make', 'Vehicle Model'],
    name = 'Serious Injuries'
    
)

gdf_2024_minor.explore(
    m = gdf_2024_serious,
    color = 'purple',
    marker_kwds = dict(radius = 3, fill = True),
    tooltip = ['Road Name', 'Weather', 'Injury Severity'],
    opacity = 0.5,
    popup = ['Crash Date/Time', 'Cross-Street Name', 'Off-Road Description', 'Vehicle Make', 'Vehicle Model'],
    name = 'Minor Injuries'
    
)


gdf_2024_possible.explore(
    m = gdf_2024_serious,
    color = 'blue',
    marker_kwds = dict(radius = 3, fill = True),
    tooltip = ['Road Name', 'Weather', 'Injury Severity'],
    opacity = 0.5,
    popup = ['Crash Date/Time', 'Cross-Street Name', 'Off-Road Description', 'Vehicle Make', 'Vehicle Model'],
    name = 'Possible Injuries'
    
)


folium.TileLayer('openstreetmap', show = True).add_to(gdf_2024_serious)
folium.LayerControl().add_to(gdf_2024_serious)
gdf_2024_serious

In [20]:
gdf_2024_serious.save('map.html')

In [21]:
# Load the dataset
df = pd.read_csv("Crash_Reporting_-_Drivers_Data.csv", low_memory=False)

# Convert Crash Date/Time to datetime
df["Crash Date/Time"] = pd.to_datetime(df["Crash Date/Time"], errors="coerce")

# Drop rows with missing values in critical columns
df = df.dropna(subset=["Crash Date/Time", "Injury Severity", "Weather"])

# Focus on Dry weather
weather_focus = "Clear"
filtered_df = df[df["Weather"] == weather_focus].copy()

# Create a new date column (datetime64[ns] for Altair compatibility)
filtered_df["Crash_Date"] = filtered_df["Crash Date/Time"].dt.floor("d")


filtered_df

C:\Users\dku19\AppData\Local\Temp\ipykernel_6480\3352931242.py:5: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Crash Date/Time"] = pd.to_datetime(df["Crash Date/Time"], errors="coerce")


,Report Number,Local Case Number,Agency Name,ACRS Report Type,Crash Date/Time,Route Type,Road Name,Cross-Street Name,Off-Road Description,Municipality,...,Speed Limit,Driverless Vehicle,Parked Vehicle,Vehicle Year,Vehicle Make,Vehicle Model,Latitude,Longitude,Location,Crash_Date
172054,HA24850015,24000424,MCPARK,Property Damage Crash,2024-03-11 11:09:00,Maryland (State) Route,FALLS RD,GLEN RD,NaN,NaN,...,35,No,NaN,2023,AUDI,A6,39.046002,-77.191530,"(39.04600169, -77.19152963)",2024-03-11
172055,MCP15500051,240016152,MONTGOMERY,Property Damage Crash,2024-04-07 09:30:00,Maryland (State) Route,CONNECTICUT AVE,NaN,NaN,NaN,...,30,No,No,2015,TOYOTA,PRIUS,38.987627,-77.077087,"(38.98762711, -77.07708655)",2024-04-07
172057,MCP3016004Z,240017335,MONTGOMERY,Property Damage Crash,2024-04-14 12:45:00,Maryland (State) Route,GERMANTOWN RD,CRYSTAL ROCK DR,NaN,NaN,...,45,No,No,2024,HYUNDAI,ELANTRA,39.181006,-77.262142,"(39.18100613, -77.26214161)",2024-04-14
172058,MCP3118004Q,240025609,MONTGOMERY,Injury Crash,2024-05-30 12:12:00,County Route,BEL PRE RD,NaN,NaN,NaN,...,35,No,NaN,2015,NISSAN,ALTIMA,39.091656,-77.069933,"(39.09165565, -77.06993308)",2024-05-30
172059,EJ7889004W,240022651,GAITHERSBURG,Injury Crash,2024-05-13 22:30:00,Municipality Route,ODENDHAL AVE (EB/L),NaN,NaN,NaN,...,30,No,No,2020,TOYOTA,COROLLA,39.148527,-77.206725,"(39.14852707, -77.20672489)",2024-05-13
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
192487,MCP2016008J,250006132,MONTGOMERY,Property Damage Crash,2025-02-11 07:14:00,Maryland (State) Route,FREDERICK RD,NaN,NaN,NaN,...,40,No,No,2007,MACK,LE,39.107325,-77.158816,"(39.10732539, -77.15881565)",2025-02-11
192488,MCP137801BX,250005146,MONTGOMERY,Property Damage Crash,2025-02-04 15:44:00,NaN,NaN,NaN,Driveway PARKING LOT CIRCLE,NaN,...,0,No,No,2023,FREIGHTLINER,B2 106 ELECTRIC CONVENTIONAL CHASSIS,39.167559,-77.210407,"(39.16755867, -77.2104069)",2025-02-04
192489,MCP3039007G,250005339,MONTGOMERY,Injury Crash,2025-02-05 15:30:00,County Route,GROVE ST,SLIGO AVE,NaN,NaN,...,35,No,No,2017,TOYOTA,RAV4,38.990191,-77.021936,"(38.9901908, -77.02193564)",2025-02-05
192491,MCP3337003Q,250003870,MONTGOMERY,Property Damage Crash,2025-01-27 15:31:00,County Route,RANDOLPH RD,NaN,NaN,NaN,...,40,No,No,2017,MASERATI,LEVANTE,39.062597,-77.041262,"(39.06259684, -77.04126238)",2025-01-27


In [ ]:
import pandas as pd
import altair as alt

#Load the data again
df = pd.read_csv("Crash_Reporting_-_Drivers_Data.csv", low_memory=False)
# convert crash date/time to datetime format
df["Crash Date/Time"] = pd.to_datetime(df["Crash Date/Time"], errors="coerce")
#clean data
df = df.dropna(subset=["Crash Date/Time", "Injury Severity", "Weather"])

#date pre-processing
df["Crash_Date"] = df["Crash Date/Time"].dt.floor("d")
df["Crash_Month"] = df["Crash_Date"].dt.to_period("M").dt.start_time

#filtering the relevant weather types and injury types
weather_options = ["Clear", "Cloudy", "Rain", "Snow"]
df = df[df["Weather"].isin(weather_options)]
df = df[df["Injury Severity"] != "No Apparent Injury"]

#grouping the data we need
grouped = (
    df.groupby(["Crash_Month", "Weather", "Injury Severity"])
    .size()
    .reset_index(name="Crash_Count")
)

#dropdown selector
weather_dropdown = alt.binding_select(
    options=weather_options,
    name="Select Weather: "
)
weather_select = alt.param(
    name="weather_param",
    bind=weather_dropdown,
    value="Clear"
)

#selection of points (hover)
hover = alt.selection_point(
    fields=["Crash_Month", "Injury Severity"],
    nearest=True,
    on="mouseover",
    empty="none",
    clear="mouseout"
)

# base chart with legend-enabled color encoding
base = alt.Chart(grouped).encode(
    x=alt.X("Crash_Month:T", title="Month"),
    y=alt.Y("Crash_Count:Q", title="Number of Crashes"),
    color=alt.Color("Injury Severity:N", title="Injury Type")
).transform_filter(
    alt.datum.Weather == weather_select
)

#Line chart
lines = base.mark_line(point=True)

#hoverable points with conditional opacity and tooltips
points = base.mark_circle(size=80).encode(
    opacity=alt.condition(hover, alt.value(1), alt.value(0)),
    tooltip=[
        alt.Tooltip("Crash_Month:T", title="Month", format="%B %Y"),
        alt.Tooltip("Injury Severity:N", title="Injury Type"),
        alt.Tooltip("Crash_Count:Q", title="Crash Count")
    ]
).add_params(
    hover
)

#combining the chart layers
chart = (lines + points).add_params(
    weather_select
).properties(
    width=750,
    height=400,
    title="Monthly Injury Crash Trends by Weather Type"
)
chart

C:\Users\dku19\AppData\Local\Temp\ipykernel_6480\78855329.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Crash Date/Time"] = pd.to_datetime(df["Crash Date/Time"], errors="coerce")


In [ ]:
chart.save("crashes_by_type.html")